<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Parse policy-mix scenario combinations and compute end-year deltas.
- **Primary inputs**: runs/policy_mix/<run_id>/*/output.csv + data/policies_scenarios_description.csv
- **Primary outputs**: data/test_policy_deltas.csv and comparison plots
- **Refactor role**: Canonical parser for policy-mix delta analysis versus Reference.
- **Data policy**: keep existing simulation data in place during refactor; no run deletion in migration phases.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Load one policy-mix run and concatenate scenario outputs.
2. Join scenario metadata from `data/policies_scenarios_description.csv`.
3. Compute end-year deltas vs `Reference` for selected indicators.
4. Generate comparison plots and export the delta table.

### Inputs
- `runs/policy_mix/<run_id>/*/output.csv`
- `data/policies_scenarios_description.csv`

### Outputs
- `data/test_policy_deltas.csv`
- Policy-mix comparison plots in the selected run folder.

### How To Reuse
- Set `folder` to a different run under `runs/policy_mix/`.


This notebook parses multiple policy mix scenarios result from Res-IRF.

In [ ]:
import os
import pandas as pd

from project.write_output import plot_compare_scenarios_simple

In [ ]:
# Refactor: default run path under scenario_analysis/runs/policy_mix
folder = os.path.join('runs', 'policy_mix', 'policies_scenarios_20230621_232329')

In [ ]:
# Read all scenarios output from a run
ignore = ['root_log.log', 'img', '.DS_Store', 'comparison.csv']
data = dict()
for result_dir in [i for i in os.listdir(folder) if i not in ignore]:
    df = pd.read_csv(os.path.join(folder, result_dir, 'output.csv'), index_col=[0])
    data.update({result_dir: df})
data = pd.concat((data), axis=0).rename_axis(['Scenarios', 'Variables'], axis=0)

In [ ]:
# Read scenarios
scenarios = pd.read_csv(os.path.join('data', 'policies_scenarios_description.csv'), index_col=[0]).rename_axis('Scenarios')

In [ ]:
df = data.join(scenarios, on='Scenarios').set_index(list(scenarios.columns), append=True)

In [ ]:
df.index

In [ ]:
end = df.columns[-1]

In [ ]:
variables = ['Ratio expenditure Single-family - Privately rented - C1 (%)', 'Cost-benefits analysis (Billion euro)']
extract = df[df.index.get_level_values('Variables').isin(variables)].loc[:, end].unstack('Variables')
reference = extract[extract.index.get_level_values('Scenarios') == 'Reference']
extract = extract[variables].sub(reference[variables].values)

In [ ]:
extract.to_csv(os.path.join('data', 'test_policy_deltas.csv'))

In [ ]:
extract[variables]

In [ ]:
reference[variables]

In [ ]:
plot_compare_scenarios_simple(data, folder, quintiles=True)